In [1]:
import cupy as cp

print(f"CuPy version: {cp.__version__}")
print(f"CUDA available: {cp.cuda.is_available()}")
print(f"Device count: {cp.cuda.runtime.getDeviceCount()}")

# Get device name
device = cp.cuda.Device(0)
print(f"Device ID: {device.id}")

# Test actual GPU computation
a = cp.array([1, 2, 3])
b = cp.array([4, 5, 6])
c = a + b
print(f"GPU computation works: {c}")
print(f"Array device: {c.device}")

CuPy version: 14.0.1
CUDA available: True
Device count: 1
Device ID: 0
GPU computation works: [5 7 9]
Array device: <CUDA Device 0>


In [2]:
N_FILTERS = 256     # Frequency bins (try 128, 256, 512)
FRMLEN_MS = 8.0       # Frame length in ms

RIPPLE_F_MIN = 1000 / (2 ** (5.3 / 2))  # ≈ 158 Hz
RIPPLE_OCTAVES = 5.3

RESOLUTION = "low"

In [3]:
import sys

sys.path.insert(0, "..")

import re
from pathlib import Path
import librosa
import time
import pygaborstm as stm

cpu_config = stm.Config(
    use_gpu=False,
    resolution=RESOLUTION,
    n_filters=N_FILTERS,
    frmlen_ms=FRMLEN_MS,
    f_min=RIPPLE_F_MIN,
    octaves=RIPPLE_OCTAVES,
)

gpu_config = stm.Config(
    use_gpu=True,
    resolution=RESOLUTION,
    n_filters=N_FILTERS,
    frmlen_ms=FRMLEN_MS,
    f_min=RIPPLE_F_MIN,
    octaves=RIPPLE_OCTAVES,
)
model_cpu = stm.PyGaborSTM(cpu_config)
model_gpu = stm.PyGaborSTM(gpu_config)

print(f"PyGaborSTM version: {stm.__version__}")

# Config
DATA_DIR = Path("./data")
RIPPLE_DIR = DATA_DIR / "mvripfft"
SR = 16000
DURATION = 5.0

print(f"RIPPLE_DIR: {RIPPLE_DIR.resolve()}")

/root/.cache/pypoetry/virtualenvs/pygaborstm-bmo-O-D_-py3.12/lib/python3.12/site-packages/cupyx/jit/_interface.py:247: FutureWarning: cupyx.jit.rawkernel is experimental. The interface can change in the future.
  cupy._util.experimental('cupyx.jit.rawkernel')


PyGaborSTM version: 0.1.0
RIPPLE_DIR: /root/JHU-LCAP/PyGaborSTM/data/mvripfft


In [4]:
def load_audio(filepath):
    """Load audio snippet."""
    audio, _ = librosa.load(filepath, sr=SR)
    max_samples = int(DURATION * SR)
    return audio[:max_samples] if len(audio) > max_samples else audio


def parse_ripple_filename(filename):
    """Extract rate and scale from filename like 'ripple_R-32.00_S0.25.wav'"""
    match = re.search(r'R(-?\d+\.?\d*)_S(\d+\.?\d*)', filename.stem)
    rate = float(match.group(1))
    scale = float(match.group(2))
    return rate, scale

def reset_gpu():
    mempool = cp.get_default_memory_pool()
    print(f"GPU memory used: {mempool.used_bytes() / 1e9:.2f} GB")
    mempool.free_all_blocks()

In [5]:
ripple_files = sorted(RIPPLE_DIR.glob("ripple_*.wav"))
print(f"Found {len(ripple_files)} ripple files")

# Get rates/scales from model config
valid_rates = set(model_cpu.config.rates)
valid_scales = set(model_cpu.config.scales)

audios = []
metadata = []

for filepath in ripple_files:
    rate, scale = parse_ripple_filename(filepath)

    # Skip if not in our config
    if rate not in valid_rates or scale not in valid_scales:
        continue

    audios.append(load_audio(filepath))

    metadata.append({"title": f"R{rate:.0f},S{scale:.1f}"})

print(f"Loaded {len(audios)} audio files")

Found 96 ripple files
Loaded 60 audio files


In [6]:
reset_gpu()

GPU memory used: 0.00 GB


In [7]:
audio = audios[0]

_ = model_cpu.rsf(model_cpu.spectrogram(audio))  # warmup

start = time.perf_counter()
spec_cpu = model_cpu.spectrogram(audio)
rsf_cpu = model_cpu.rsf(spec_cpu)
cpu_time = time.perf_counter() - start
print(f"CPU: {cpu_time:.3f}s")

CPU: 3.755s


In [8]:
# GPU
_ = model_gpu.rsf(model_gpu.spectrogram(audio))  # warmup

start = time.perf_counter()
spec_gpu = model_gpu.spectrogram(audio)
rsf_gpu = model_gpu.rsf(spec_gpu)
gpu_time = time.perf_counter() - start
print(f"GPU: {gpu_time:.3f}s")

print(f"Speedup: {cpu_time / gpu_time:.1f}x")

GPU: 0.421s
Speedup: 8.9x


In [9]:
# After GPU benchmark
mempool = cp.get_default_memory_pool()
print(f"GPU memory used: {mempool.used_bytes() / 1e9:.2f} GB")
mempool.free_all_blocks()

GPU memory used: 0.40 GB


In [10]:
# Full benchmark with all 60 files
start = time.perf_counter()
for audio in audios:
    spec = model_cpu.spectrogram(audio)
    rsf = model_cpu.rsf(spec)
cpu_time = time.perf_counter() - start
print(f"CPU: {cpu_time:.3f}s")

CPU: 248.536s


In [11]:
start = time.perf_counter()
for audio in audios:
    spec = model_gpu.spectrogram(audio)
    rsf = model_gpu.rsf(spec)
gpu_time = time.perf_counter() - start

print(f"GPU: {gpu_time:.3f}s")
print(f"Speedup: {cpu_time / gpu_time:.1f}x")

GPU: 25.706s
Speedup: 9.7x


In [12]:
start = time.perf_counter()
specs = [model_cpu.spectrogram(audio) for audio in audios]
print(f"CPU spectrogram: {time.perf_counter() - start:.3f}s")

start = time.perf_counter()
rsfs = [model_cpu.rsf(spec) for spec in specs]
print(f"CPU RSF: {time.perf_counter() - start:.3f}s")

CPU spectrogram: 65.057s
CPU RSF: 183.170s


In [13]:
# Time them separately
start = time.perf_counter()
specs = [model_gpu.spectrogram(audio) for audio in audios]
print(f"GPU spectrogram: {time.perf_counter() - start:.3f}s")

start = time.perf_counter()
rsfs = [model_gpu.rsf(spec) for spec in specs]
print(f"GPU RSF: {time.perf_counter() - start:.3f}s")

GPU spectrogram: 22.334s
GPU RSF: 4.515s
